# 07 Hyperparameter Tuning with Optuna
This notebook uses Optuna to find the best hyperparameters for the pneumonia classifier. 

**Update**: Since we hit Recall=1.0 easily, we are now optimizing for **F1-Score** to ensure a better balance between Precision and Recall.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import optuna
from optuna.trial import TrialState
from sklearn.metrics import recall_score, f1_score
import numpy as np

from pneumonia_classifier.ml.model.arch import Net

## Configuration and Data Loading

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = r'j:\Users\ayush\Desktop\code\pneumonia_classifier\artifacts\02_12_2025_08_52_04\data_ingestion\data\data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
INPUT_SIZE = (224, 224)

def get_loaders(batch_size):
    transform = transforms.Compose([
        transforms.Resize(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
    test_dataset = datasets.ImageFolder(TEST_DIR, transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader

## Objective Function
Optimizing **F1-Score** to handle class imbalance more effectively.

In [ ]:
def objective(trial):
    # Hyperparameters to tune
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    
    model = Net().to(DEVICE)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    
    train_loader, test_loader = get_loaders(batch_size)
    
    EPOCHS = 5
    for epoch in range(EPOCHS):
        model.train()
        for data, target in train_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()
            
        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(DEVICE), target.to(DEVICE)
                output = model(data)
                pred = output.argmax(dim=1).cpu().numpy()
                all_preds.extend(pred)
                all_targets.extend(target.cpu().numpy())
        
        # Using F1-Score instead of Recall
        score = f1_score(all_targets, all_preds, pos_label=1)
        
        trial.report(score, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
    return score

## Run Optimization

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(study.get_trials(states=(TrialState.PRUNED,))))
print("  Number of complete trials: ", len(study.get_trials(states=(TrialState.COMPLETE,))))

print("Best trial:")
trial = study.best_trial

print("  F1-Score: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

## Save Best Model

In [ ]:
if 'study' in locals():
    # Re-train the model with best parameters and save it
    best_params = study.best_params
    model = Net().to(DEVICE)
    optimizer = getattr(optim, best_params['optimizer'])(model.parameters(), lr=best_params['lr'])
    train_loader, _ = get_loaders(best_params['batch_size'])

    EPOCHS = 10 
    print(f"Re-training model with best params: {best_params}")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for data, target in train_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch}/{EPOCHS} complete")

    torch.save(model.state_dict(), 'best_model_optuna.pt')
    print("Best model saved as best_model_optuna.pt")
else:
    print("Optimization study not found in memory. Skip re-training.")

## Load and Verify Saved Model
This section is independent of the study variables. You can run this directly if `best_model_optuna.pt` exists.

In [ ]:
MODEL_PATH = 'best_model_optuna.pt'
VERIFY_BATCH_SIZE = 8

if os.path.exists(MODEL_PATH):
    # Load for verification
    loaded_model = Net().to(DEVICE)
    loaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    loaded_model.eval()
    print(f"Loaded model from {MODEL_PATH}")

    _, test_loader = get_loaders(VERIFY_BATCH_SIZE)
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = loaded_model(data)
            pred = output.argmax(dim=1).cpu().numpy()
            all_preds.extend(pred)
            all_targets.extend(target.cpu().numpy())

    final_f1 = f1_score(all_targets, all_preds, pos_label=1)
    final_recall = recall_score(all_targets, all_preds, pos_label=1)
    print(f"Verified Model Performance:")
    print(f"  F1 Score: {final_f1:.4f}")
    print(f"  Recall:   {final_recall:.4f}")
else:
    print(f"Error: {MODEL_PATH} not found. Please run the training cell above first.")

## Visualizations

In [8]:
if 'study' in locals():
    try:
        from optuna.visualization import plot_optimization_history, plot_param_importances
        fig1 = plot_optimization_history(study)
        fig1.show()
        
        fig2 = plot_param_importances(study)
        fig2.show()
    except Exception as e:
        print(f"Visualization error: {e}")
else:
    print("Optimization study not found in memory. Visualizations are only available immediately after running the optimization.")

Optimization study not found in memory. Visualizations are only available immediately after running the optimization.
